# Complete Guide to Evaluator

`Evaluator` is the abstract base class used to measure algorithm quality in AlgoDisco. Every custom evaluator must inherit from it and implement `evaluate_program`.


## 1. EvalResult Type Definition


In [ ]:
from algodisco.base.evaluator import EvalResult

result: EvalResult = {
    "score": 0.95,
    "execution_time": 0.023,
    "error_msg": None
}

print(f"score: {result['score']}")
print(f"execution_time: {result.get('execution_time')}")
print(f"error_msg: {result.get('error_msg')}")

## 2. Evaluator Base Class


In [ ]:
from algodisco.base.evaluator import Evaluator, EvalResult
import inspect

print("Evaluator.evaluate_program signature:")
print(inspect.signature(Evaluator.evaluate_program))


## 3. Creating a Custom Evaluator


In [ ]:
import random
import subprocess
import tempfile

class SortingEvaluator(Evaluator):
    def __init__(self, num_tests=100):
        self.num_tests = num_tests
        self.test_cases = self._generate_test_cases()
    
    def _generate_test_cases(self):
        cases = []
        for _ in range(self.num_tests):
            n = random.randint(0, 20)
            case = random.sample(range(-100, 100), n)
            expected = sorted(case)
            cases.append((case, expected))
        return cases
    
    def evaluate_program(self, program_str: str) -> EvalResult:
        try:
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name

            correct = 0
            for inputs, expected in self.test_cases[:10]:
                test_code = f"""import sys; sys.path.insert(0, '.'); exec(open('{temp_path}').read()); result = sort_list({inputs}); print(result == {expected})"""
                exec_result = subprocess.run(
                    ['python', '-c', test_code],
                    capture_output=True,
                    timeout=5
                )
                if exec_result.returncode == 0 and 'True' in exec_result.stdout.decode():
                    correct += 1

            score = correct / min(10, len(self.test_cases))

            return {
                "score": score,
                "execution_time": 0.0,
                "error_msg": None
            }

        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)[:200]
            }

evaluator = SortingEvaluator(num_tests=20)
test_program = '''
def sort_list(lst):
    return sorted(lst)
'''
result = evaluator.evaluate_program(test_program)
print(f"Score: {result['score']}")

## 4. Best Practices


In [ ]:
class SafeEvaluator(Evaluator):
    def evaluate_program(self, program_str: str) -> EvalResult:
        try:
            compile(program_str, '<string>', 'exec')
            score = self._run_tests(program_str)
            return {"score": score, "execution_time": 0.0, "error_msg": None}
        except SyntaxError as e:
            return {"score": 0.0, "execution_time": 0.0, "error_msg": "Syntax Error: " + str(e)}
        except Exception as e:
            return {"score": 0.0, "execution_time": 0.0, "error_msg": "Error: " + str(e)}
    
    def _run_tests(self, program_str):
        return 0.5

eval_safe = SafeEvaluator()
result = eval_safe.evaluate_program("x = 1")
print(result)

In [ ]:
from typing import TypedDict, Optional

class DetailedEvalResult(TypedDict):
    score: float
    execution_time: Optional[float]
    error_msg: Optional[str]
    test_passes: int
    test_total: int

class DetailedEvaluator(Evaluator[DetailedEvalResult]):
    def __init__(self, num_tests=100):
        self.num_tests = num_tests
    
    def evaluate_program(self, program_str: str) -> DetailedEvalResult:
        passes = min(self.num_tests, 85)
        return {
            "score": passes / self.num_tests,
            "execution_time": 0.1,
            "error_msg": None,
            "test_passes": passes,
            "test_total": self.num_tests
        }

detailed_eval = DetailedEvaluator(num_tests=100)
result = detailed_eval.evaluate_program("code")
print(f"Score: {result['score']}, Passes: {result['test_passes']}/{result['test_total']}")

## Summary

| Key Point | Description |
|--------|-------|
| `evaluate_program` | The only required method |
| Return type | Must be a dictionary that contains `score` |
| Error handling | Always return a valid result |
| Performance | Use timeouts or sandboxing when needed |
